# Web of Science Collection Pipeline

*Author: Regina Chua*

> This notebook is the Web of Science (WoS) arm of the systematic review. WoS offers a REST API
> (Clarivate), so — like PubMed and SCOPUS — collection can be fully scripted. It still needs an
> **institutional API key**, which is the outstanding blocker on the roadmap. The pipeline reuses the
> shared `(disease) AND (spatial) AND (exposure) NOT (exclusions)` criteria from
> `search_strategy.py`, translated into WoS' `TS=()` topic syntax.

This uses the **Web of Science Starter API** (`wos-starter/v1`). The Starter API returns metadata
(title, authors, journal, year, DOI, keywords, citation counts) but **not abstracts** — those
require the Expanded API. The key is read from the environment (`WOS_API_KEY`) so it never lands in
the notebook or git.

I put in an API application here: https://developer.clarivate.com/apis/wos

**References:** [WoS Starter API docs](https://developer.clarivate.com/apis/wos-starter).

## 1. Environment Setup

> Imports plus the API key. I load `WOS_API_KEY` from `.env` (same pattern as `PUBMED_EMAIL` in the
> PubMed notebook) and only flip `WOS_READY` to True when it's present, so the collection cell can
> skip gracefully while we're still waiting on institutional access. `requests` is the only extra
> dependency.

In [ ]:
import os
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

from search_strategy import (
    INCLUSION_CRITERIA,
    ALTERNATE_TERMS,
    EXCLUSION_TERMS,
    DATE_FILTER,
    CLEANING_RULES,
)

load_dotenv()
pd.set_option("display.max_colwidth", 120)

WOS_API_KEY = os.getenv("WOS_API_KEY")
WOS_READY = bool(WOS_API_KEY)
print("Environment ready. WOS_API_KEY set:", WOS_READY)
if not WOS_READY:
    print("Add WOS_API_KEY=... to your .env to enable collection (institutional access required).")

## 2. Build Query

> WoS' topic operator `TS=()` searches title, abstract, author keywords, and Keywords Plus — the
> close analogue of PubMed's `[TIAB]` and SCOPUS' `TITLE-ABS-KEY()`. I build three `AND`-joined
> `TS=()` groups and append `NOT TS=()` for the exclusions, then constrain the year with `PY=`. WoS
> supports the `*` wildcard, so the existing terms transfer directly.

In [ ]:
def merge_terms(primary, alternates):
    """Combine primary + alternate/NLP terms per category (order-preserving,
    case-insensitive de-duplication). Mirrors pubmed.ipynb."""
    merged = {}
    for category, base_terms in primary.items():
        seen, combined = set(), []
        for t in list(base_terms) + list(alternates.get(category, [])):
            if t.lower() not in seen:
                seen.add(t.lower())
                combined.append(t)
        merged[category] = combined
    return merged


def ts_group(terms):
    """Wrap terms in a WoS TS=() topic OR-group."""
    return "TS=(" + " OR ".join(f'\"{t}\"' for t in terms) + ")"


def build_wos_query(inclusion, exclusion, date_filter):
    """Compose the WoS query.

    Structure: TS=(disease) AND TS=(spatial) AND TS=(exposure)
               NOT TS=(exclusions) AND PY=(start-end)
    """
    include = " AND ".join([
        ts_group(inclusion["disease"]),
        ts_group(inclusion["spatial"]),
        ts_group(inclusion["exposure"]),
    ])
    q = f"{include} NOT {ts_group(exclusion)}"
    start_year = int(date_filter["start_date"][:4])
    end_year = int(date_filter["end_date"][:4]) if date_filter.get("end_date") else datetime.now().year
    q += f" AND PY=({start_year}-{end_year})"
    return q


INCLUDE_ALTERNATE_TERMS = True
search_criteria = (
    merge_terms(INCLUSION_CRITERIA, ALTERNATE_TERMS)
    if INCLUDE_ALTERNATE_TERMS
    else INCLUSION_CRITERIA
)

query = build_wos_query(search_criteria, EXCLUSION_TERMS, DATE_FILTER)

print("Alternate terms folded into query:", INCLUDE_ALTERNATE_TERMS)
for category, term_list in search_criteria.items():
    print(f"{category} terms ({len(term_list)}):", term_list)
print("\nQuery:\n", query)

## 3. Collect Articles from Web of Science

> This pages through the Starter API's `/documents` endpoint, 50 records at a time, sending the key
> in the `X-ApiKey` header. I cap the run with `MAX_RECORDS` and pause briefly between pages to stay
> under the rate limit. Each hit is flattened into the shared schema (`source = "web_of_science"`).
> Abstracts come back empty under the Starter API — see the notes in Section 6.

In [ ]:
WOS_BASE = "https://api.clarivate.com/apis/wos-starter/v1/documents"
PAGE_SIZE = 50      # API max per page
MAX_RECORDS = 1000  # safety cap for a single run
SLEEP_BETWEEN_PAGES = 1.0


def _first_doi(identifiers):
    return identifiers.get("doi") if isinstance(identifiers, dict) else None


def _citation_count(hit):
    """Sum citation counts across the databases WoS reports them for."""
    cites = hit.get("citations") or []
    try:
        return sum(c.get("count", 0) for c in cites)
    except Exception:
        return None


def flatten_wos(hit):
    """Flatten a Starter API document into our shared schema."""
    source = hit.get("source", {}) or {}
    names = (hit.get("names", {}) or {}).get("authors", []) or []
    identifiers = hit.get("identifiers", {}) or {}
    keywords = (hit.get("keywords", {}) or {}).get("authorKeywords", []) or []
    doi = _first_doi(identifiers)
    year = source.get("publishYear")
    return {
        "title":            hit.get("title"),
        "abstract":         None,  # not available in the Starter API
        "publication_date": f"{year}-01-01" if year else None,
        "authors":          "; ".join(a.get("displayName", "") for a in names),
        "journal":          source.get("sourceTitle"),
        "doi":              doi,
        "url":              f"https://doi.org/{doi}" if doi else None,
        "num_citations":    _citation_count(hit),
        "pubmed_id":        identifiers.get("pmid"),
        "keywords":         keywords,
        "source":           "web_of_science",
    }


def fetch_wos(query, api_key, max_records=MAX_RECORDS):
    headers = {"X-ApiKey": api_key, "Accept": "application/json"}
    records, page = [], 1
    while len(records) < max_records:
        params = {"db": "WOS", "q": query, "limit": PAGE_SIZE, "page": page}
        resp = requests.get(WOS_BASE, headers=headers, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        hits = payload.get("hits", [])
        if not hits:
            break
        records.extend(flatten_wos(h) for h in hits)
        total = payload.get("metadata", {}).get("total", len(records))
        print(f"  page {page}: +{len(hits)} ({len(records)}/{total})")
        if len(records) >= total:
            break
        page += 1
        time.sleep(SLEEP_BETWEEN_PAGES)
    return records


run_ts = datetime.now().isoformat(timespec="seconds")
df_raw = pd.DataFrame()

if not WOS_READY:
    print("Skipping collection — WOS_API_KEY is not set (see Section 1).")
else:
    try:
        print(f"Querying Web of Science at {run_ts} ...")
        df_raw = pd.DataFrame(fetch_wos(query, WOS_API_KEY))
        print(f"\nCollected {len(df_raw)} records.")
    except requests.HTTPError as exc:
        print(
            f"WoS API returned an error: {exc}\n"
            "Check the key is valid and entitled to the Starter API, and that the query is well-formed."
        )
    except Exception as exc:  # noqa: BLE001
        print(f"WoS request failed: {type(exc).__name__}: {exc}")

preview_cols = [c for c in ["title", "publication_date", "journal", "doi", "num_citations"]
                if c in df_raw.columns]
if not df_raw.empty:
    display(df_raw[preview_cols].head())

## 4. Clean Results

> Same cleaning contract as the other databases, via `CLEANING_RULES`: drop rows without a title,
> deduplicate on normalised title, and require a DOI by default for traceability in the cross-
> database merge.

In [ ]:
df_clean = df_raw.copy()

if not df_clean.empty:
    df_clean = df_clean.dropna(subset=["title"])
    if CLEANING_RULES.get("remove_duplicate_titles", True):
        df_clean["_title_lower"] = df_clean["title"].astype(str).str.lower().str.strip()
        df_clean = df_clean.drop_duplicates(subset=["_title_lower"]).drop(columns=["_title_lower"])
    if CLEANING_RULES.get("require_doi", True) and "doi" in df_clean.columns:
        df_clean = df_clean.dropna(subset=["doi"])

print(f"Records after cleaning: {len(df_clean)}  (from {len(df_raw)} raw)")
if not df_clean.empty:
    display(df_clean[preview_cols].head())

## 5. Export

> Write the cleaned set to CSV with the shared column layout for the Milestone 4 deduplication merge.
> Only writes when there is data, so a run blocked on access doesn't clobber a previous export.

In [ ]:
output_path = Path("web_of_science_results_2026.csv")

if df_clean.empty:
    print("Nothing to export — df_clean is empty (see Sections 1 and 3).")
else:
    df_clean.to_csv(output_path, index=False)
    print(f"Exported {len(df_clean)} records to {output_path.resolve()}")
    print(f"Run timestamp: {run_ts}")

## 6. Notes & Next Steps

> - **Access:** request a Web of Science API key through the institution's Clarivate subscription and
>   add it as `WOS_API_KEY` in `.env`. This is the roadmap blocker for WoS.
> - **Abstracts:** the Starter API does not return abstract text. If abstracts are needed for LLM
>   pre-screening (Milestone 2), either upgrade to the **Expanded API** (`wos/v1`, richer records) or
>   backfill abstracts via DOI from another source during deduplication.
> - **Manual fallback:** without API access, the WoS web UI can export the same `TS=()` search to RIS;
>   the loader in `embase.ipynb` reads RIS and can be reused (just change `source` to
>   `web_of_science`).
> - **Schema:** export columns match the other databases so all four sources concatenate cleanly in
>   Milestone 4.